In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Mixture of Experts

Every transformer of this chapter so far is *dense*: each token passes
through every parameter. A forward pass costs about two floating-point
operations per parameter and token (that section), so a model with
twice the parameters, which can store more of what it reads, also costs
twice as much to train and to run. Yet most of what a large model knows is
of no use for any one token. Continuing "the integral of" draws on
different knowledge from continuing "Act I, Scene". Can a model hold more
parameters without spending more computation on each token?

A *mixture of experts* (MoE) does this by running only part of the model
for each token. It keeps $E$ copies of each block's feed-forward network,
the *experts*, and a small learned *router* sends each token to only $k$
of them. The stored parameters then grow with $E$, while the computation
per token grows only with $k$. DeepSeek-V3, for example, stores 671 billion
parameters and uses 37 billion of them for each token
[@Liu.Feng.Xue.ea.2024].

The idea is old. @Jacobs.Jordan.Nowlan.ea.1991 trained a committee
of specialist networks under a gating network, and
@Shazeer.Mirhoseini.Maziarz.ea.2017 made the gate sparse, so that
only a few of thousands of experts run for each input. This section counts
stored and active parameters, implements the layer, and studies how it
fails: a router left to itself sends most tokens to a few experts. We
compare two remedies, an auxiliary loss and a controller that adds no
gradient. Finally, we give the GPT of that section eight experts per
block and train it against the dense model at equal computation per token.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## Conditional Computation

The feed-forward network is the natural place for experts. A block of width
$d$ spends $4d^2$ parameters on the four attention projections and, with
the SwiGLU network of that section (three matrices of
size $d \times 8d/3$), $8d^2$ on the feed-forward network: two thirds of
the block. And while attention mixes information between positions, the
feed-forward network transforms each position on its own, so one token can
be handed to a different network than its neighbor without affecting it.

Replacing the network by $E$ experts, of which each token uses $k$,
multiplies the stored feed-forward parameters by $E$ and the active ones by
$k$, and adds a router, a $d \times E$ matrix per layer. At about two
operations per active parameter and token, the cost of a token follows the
active count. The function below counts both for a stack of expert layers.
We apply it to two deployed models and to our own configuration, the one
trained at the end of this section. Mixtral 8x7B has 8 full-width experts
per layer, of which each token uses 2 [@Jiang.Sablayrolles.Roux.ea.2024].
DeepSeek-V3 has 256 narrow experts per layer, of which each token uses 8,
plus one *shared* expert that every token uses [@Liu.Feng.Xue.ea.2024].

In [ ]:
def moe_accounting(name, d, width, num_experts, num_active, num_layers,
                   shared=0, unit=1e9):
    """Expert parameters stored, and expert and router parameters active
    per token, in a stack of SwiGLU expert layers."""
    per_expert = 3 * d * width          # SwiGLU: three d x width matrices
    store = num_layers * (num_experts + shared) * per_expert
    active = num_layers * ((num_active + shared) * per_expert
                           + d * num_experts)      # + the router
    tag = 'B' if unit == 1e9 else 'M'
    print(f'{name:>12}: {store/unit:6.1f}{tag} in experts, '
          f'{active/unit:5.1f}{tag} active per token, '
          f'ratio {store/active:4.1f}x')
    return store, active

moe_accounting('ours', 256, 683, 8, 1, 6, unit=1e6)
moe_accounting('DeepSeek-V3', 7168, 2048, 256, 8, 58, shared=1)
store, active = moe_accounting('Mixtral 8x7B', 4096, 14336, 8, 2, 32)
attn = 32 * (2 * 4096 * 4096 + 2 * 4096 * 1024)  # GQA: 32 Q, 8 KV heads
emb = 2 * 32000 * 4096                           # embedding + output head
print(f'{"":>12}  with attention and embeddings: '
      f'{(store + attn + emb)/1e9:.1f}B stored, '
      f'{(active + attn + emb)/1e9:.1f}B active')

DeepSeek-V3's 58 expert layers store 656B of its 671B parameters, and a
token uses 23B of them: the layers store 28 times what a token uses. (The
37B on its model card add
attention, the embeddings, and three dense layers, which our count of the
expert layers leaves out.) For Mixtral, adding the attention layers, which
share each key and value head among four query heads (that section),
and the separate embedding and output matrices gives 46.7B stored and 12.9B
active parameters. The name "8x7B" thus describes what the model stores,
not the work per token: a token uses less than two sevenths of the
parameters. Our configuration has Mixtral's eight experts per layer but
sends each token to one of them, so it stores eight times what it uses.

The count rests on two assumptions. First, it counts only computation.
All $E$ experts must sit in accelerator memory, and during training each
of them receives gradients from the tokens routed to it. Large models
therefore spread their experts over many devices and send every token to
its experts and back, a systems problem that the Language Models part
takes up. Second, the count assumes that the router spreads tokens evenly
over the experts. If it sends most tokens to a few of them, the other
experts hold parameters that hardly ever run, and the model behaves like a
smaller one.

## A Mixture-of-Experts Layer

The layer replaces the feed-forward network of a block, and nothing else:
attention, normalization, and the residual stream stay as they are. We
first define what it computes and then how to compute it efficiently.

### Top-$k$ Routing

A linear router scores the $E$ experts for a token
$\mathbf{x} \in \mathbb{R}^d$, the $k$ largest scores select experts, and
the outputs of the selected experts are added, each weighted by the
router's probability:

$$
\mathrm{MoE}(\mathbf{x}) = \sum_{i \in \mathcal{E}_k(\mathbf{x})} p_i(\mathbf{x})\, \mathrm{FFN}_i(\mathbf{x}),
\qquad
\mathbf{p}(\mathbf{x}) = \mathrm{softmax}(\mathbf{W}_r \mathbf{x}),
$$

where $\mathbf{W}_r \in \mathbb{R}^{E \times d}$ is the router and
$\mathcal{E}_k(\mathbf{x})$ holds the indices of the $k$ largest entries of
$\mathbf{p}(\mathbf{x})$. The layer is thus a feed-forward network chosen
per token: two tokens of the same sentence may use entirely different
parameters. Because each token chooses its experts without regard to the
others, this is called *token-choice* routing.
the figure follows one token through a layer with $E = 8$ and
$k = 2$.

![A mixture-of-experts layer with eight experts, two of which are active per token. The router scores every expert. The two most probable experts run, and their outputs are weighted by their probabilities and added. The other six experts are stored but do no work for this token.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-transformers-moe.svg)

The weight of a selected expert is its probability $p_i$ itself, not a
probability renormalized over the $k$ selected experts. This choice
follows the Switch transformer [@fedus2022switch]. Selecting the top
$k$ is a discrete decision and has no gradient. The weight of a selected
expert has one: if a larger contribution of expert $i$ would lower the
loss, the gradient raises $p_i$, and with it the chance that the router
selects expert $i$ for similar tokens. Through the softmax, every logit
receives a gradient, including those of the experts that were not
selected. Mixtral instead renormalizes the weights of the selected experts
to sum to one [@Jiang.Sablayrolles.Roux.ea.2024]. The router then
learns only how to divide the weight among the selected experts, and for
top-1 routing, where the single weight is always one, it learns nothing
from the loss at all. With raw probabilities, a top-1 router still
learns. The first exercise works out these gradients.

### Dispatching Tokens to Experts

Evaluating the equation literally, by running all $E$ experts on
every token and multiplying the outputs of the unselected ones by zero,
gives the right answer at $E$ times the cost and saves nothing. An
efficient layer brings each token to its experts instead. A minibatch of
$n$ tokens makes $nk$ assignments, pairs of a token and one of its
experts, and how many of them each expert receives changes from batch to
batch. Matrix
multiplications need fixed shapes, so each expert gets a buffer of
$C = \lceil c\, nk/E \rceil$ slots, where the *capacity factor* $c$ leaves
room for an uneven spread; the Switch transformer trains with $c = 1.25$
[@fedus2022switch].

We number the assignments of each expert in the
order of the tokens and copy the token of an expert's $r^\textrm{th}$
assignment into slot $r$ of that expert's buffer. An assignment with
$r \geq C$ is *dropped*: that expert skips the token, which keeps the
outputs of its other experts and, through the residual connection, its
input. All $E$ experts then run at once, as one batched matrix
multiplication over buffers of shape $(E, C, d)$, and each token collects
its $k$ outputs, weighted by their probabilities. The work is that of
$EC \approx cnk$ rows, a fraction $ck/E$ of running every expert on every
token. Fixed shapes come with dropping: a balanced router loses few
assignments, an unbalanced one many.

Besides its output, the layer keeps four things for the experiments
below, which its method `record` updates for every batch: `expert_bias`,
a vector added to the probabilities for the top-$k$ selection only, which
stays zero until we need it; `load`, the number of assignments each expert
has received, dropped or not; `dropped`, the number of dropped
assignments; and `aux_loss`, a measure of imbalance that we derive when we
use it.

In [ ]:
class MoELayer(nnx.Module):
    """Token-choice top-k mixture of SwiGLU d2l.FeedForward experts, each
    with a buffer of C = capacity_factor * n k / E slots for n tokens."""
    def __init__(self, num_hiddens, num_experts, num_active,
                 capacity_factor=1.25, *, rngs):
        self.num_experts, self.num_active = num_experts, num_active
        self.capacity_factor = capacity_factor
        self.router = d2l.Linear(num_hiddens, num_experts, use_bias=False,
                                 rngs=rngs)
        self.experts = nnx.List([d2l.FeedForward(num_hiddens, rngs=rngs)
                                 for _ in range(num_experts)])
        self.expert_bias = nnx.Variable(jnp.zeros(num_experts))
        self.load = nnx.Variable(jnp.zeros(num_experts))
        self.dropped = nnx.Variable(jnp.zeros(()))
        self.aux_loss = nnx.Variable(jnp.zeros(()))

    def __call__(self, X):
        x = X.reshape(-1, X.shape[-1])                 # (n, d): the tokens
        (n, d), E, k = x.shape, self.num_experts, self.num_active
        C = math.ceil(self.capacity_factor * n * k / E)  # slots per expert
        P = jax.nn.softmax(self.router(x), -1)         # (n, E)
        _, idx = jax.lax.top_k(P + self.expert_bias[...], k)  # (n, k)
        e = idx.ravel()                                # expert of each pair
        onehot = jax.nn.one_hot(e, E, dtype=jnp.int32)  # (n k, E)
        rank = (onehot.cumsum(0) * onehot).sum(1) - 1  # place in its queue
        slot = jnp.where(rank < C, e * C + rank, E * C)  # E C: dropped
        buf = jnp.zeros((E * C, d), x.dtype).at[slot].set(
            jnp.repeat(x, k, 0), mode='drop')
        Y = self.run_experts(buf.reshape(E, C, d)).reshape(E * C, d)
        Y = Y.at[slot].get(mode='fill', fill_value=0)  # dropped: zeros
        weights = jnp.take_along_axis(P, idx, -1)      # (n, k): the p_i
        out = (weights[..., None] * Y.reshape(n, k, d)).sum(1)
        self.record(onehot.sum(0), (rank >= C).sum(), P)
        return out.reshape(X.shape)

    def run_experts(self, H):
        """Each expert's SwiGLU on its own buffer: (E, C, d) -> (E, C, d)."""
        W = lambda name: jnp.stack(  # (E, d_in, d_out): one per expert
            [getattr(e, name).kernel[...] for e in self.experts])
        return (jax.nn.silu(H @ W('W_g')) * (H @ W('W_1'))) @ W('W_2')

    def record(self, counts, num_dropped, P):
        """Add a batch's assignments per expert and drops to the totals,
        and keep its balancing loss."""
        self.load[...] += counts
        self.dropped[...] += num_dropped
        f = counts / counts.sum()                      # this batch's shares
        self.aux_loss[...] = self.num_experts * (f * P.mean(0)).sum()

    def reset_stats(self):
        self.load[...] = jnp.zeros_like(self.load[...])
        self.dropped[...] = jnp.zeros(())

The cumulative sum over the one-hot encoding numbers the assignments of
each expert: row $j$ counts, for every expert, the assignments to it among
the first $j$, so its entry at the expert of assignment $j$ is that
assignment's place in its expert's queue, counted from one. Slot $EC$ lies
past the end of the buffers and takes every dropped assignment.
`run_experts` stacks one of the three matrices of every `d2l.FeedForward`
into an array of shape
$(E, d, w)$ or $(E, w, d)$, for experts of width $w$, and applies the
SwiGLU of that section as three batched products, one
matrix per buffer. It reads the experts' `W_g`, `W_1`, and `W_2` by name,
so it assumes their default SwiGLU without biases.

Under `jit`, every array needs a shape that is known when the function is
compiled. Here every shape follows from the shape of `X`, since $C$
depends on $n$ only; the routing decides which rows go where, not how many
there are. Indexing with `mode='drop'` skips the updates whose index lies
past the end of the array, the dropped assignments, and `mode='fill'` reads
zeros for them.

Two helpers find the expert layers of a model and count the parameters that
one token uses, which are all parameters except the unselected experts:

In [ ]:
def moe_layers(model):
    """The MoELayer modules of a model."""
    return [m for _, m in nnx.iter_graph(model) if isinstance(m, MoELayer)]

def active_params(model):
    """Parameters used per token: all but the unselected experts."""
    return d2l.num_params(model) - sum(
        d2l.num_params(l.experts) * (l.num_experts - l.num_active)
        // l.num_experts for l in moe_layers(model))

Since the layer maps $(n, d)$ to $(n, d)$ like any feed-forward network, it
fits into `d2l.TransformerBlock` through the block's `ffn_factory`
argument, which builds the feed-forward module of the block. We check
shapes on 400 random tokens, count parameters for eight experts with two
active, and look at how the router of the fresh layer spreads the 800
assignments over buffers of $\lceil 1.25 \cdot 800/8 \rceil = 125$ slots:

In [ ]:
moe = MoELayer(256, num_experts=8, num_active=2,
               rngs=nnx.Rngs(d2l.get_key()))
X = jax.random.normal(d2l.get_key(), (4, 100, 256))
d2l.check_shape(moe(X), X.shape)
blk = d2l.TransformerBlock(
    256, num_heads=8, rngs=nnx.Rngs(d2l.get_key()),
    ffn_factory=lambda rngs: MoELayer(256, 8, 2, rngs=rngs))
d2l.check_shape(blk(X), X.shape)
print(f'{d2l.num_params(moe)} parameters stored, '
      f'{active_params(moe)} active per token')
print('shares of the assignments:',
      ' '.join(f'{s:.2f}' for s in (moe.load / moe.load.sum()).tolist()))
print(f'{float(moe.dropped) / float(moe.load.sum()):.0%} of them dropped')

Each expert holds $3 \cdot 256 \cdot 683 = 524{,}544$ parameters, so the
layer stores about 4.2 million and a token uses about 1.05 million, a
quarter. A fresh router spreads random tokens roughly evenly: no expert's
share is far from the even share of $1/8$, and the buffers, a quarter
larger than an even share, drop almost no assignments. Random vectors are
not text, however, and the router has yet to be trained. The next part
follows the shares inside a language model.

## Routing Collapse and Load Balancing

The router and the experts learn together, and the way they learn tends to
concentrate the tokens on a few experts. This part explains why and
compares two remedies.

### Positive Feedback in Routing

Follow the feedback loop. At initialization some expert is, by chance,
slightly better than its peers on the tokens it receives. The loss
gradient then rewards it twice: its parameters improve on those tokens,
*and* the router's gradient raises its probability, because weighting a
helpful expert more heavily lowers the loss. More tokens bring more
gradient signal, which makes a better expert, which attracts more tokens.
The experts that lose the early rounds receive ever fewer tokens, learn
ever more slowly, and lose probability. This loop can end in *routing
collapse*: a few experts receive most of the assignments, and much of the
stored capacity goes unused. @Shazeer.Mirhoseini.Maziarz.ea.2017
observed this imbalance and added balancing terms to the training loss. We
describe two remedies in use today.

### An Auxiliary Balancing Loss

The classic remedy adds a differentiable penalty for imbalance to the
training loss. For a minibatch, let $f_i$ be the fraction of the
assignments that expert $i$ received. It is a count, so no gradient flows
through it. Let $\bar{p}_i$ be the mean of the router's probability
$p_i(\mathbf{x})$ over the tokens of the minibatch. The loss of the Switch
transformer [@fedus2022switch], which descends from GShard's
[@Lepikhin.Lee.Xu.ea.2021], multiplies the two:

$$
\mathcal{L}_{\mathrm{balance}} = E \sum_{i=1}^{E} f_i\, \bar{p}_i.
$$

The loss is large when the experts that receive many tokens also receive
high probability. For uniform routing, $f_i = \bar{p}_i = 1/E$ and the
loss equals $1$. If one expert receives every token with probability
close to one, it approaches $E$.

The balancing loss is added to the language-modeling loss with a small
weight $\alpha$, 0.01 in the Switch transformer. Its gradient,
which flows through $\bar{p}$ only, lowers the probability of overloaded
experts for every token, whatever the token needs. This is its drawback:
the balancing gradient is added to the language-modeling gradient at every
step, and the two conflict when some tokens do need a busy expert. With
$\alpha$ too large, routing serves balance rather than prediction; with
$\alpha$ too small, the imbalance persists [@Wang.Gao.Zeng.ea.2024].

### Balancing Without a Loss

The conflict disappears if balance acts on the selection of experts alone
and adds nothing to the loss. @Wang.Gao.Zeng.ea.2024 balance the
load this way, and DeepSeek-V3 trains with their method
[@Liu.Feng.Xue.ea.2024]. A bias $b_i$ per expert is added to the
probabilities for the selection, and only for the selection. After each
optimizer step, a controller moves each bias by a fixed amount $u$ toward
balance:

$$
\mathcal{E}_k(\mathbf{x}) = \operatorname{argtop}_k \big(p_i(\mathbf{x}) + b_i\big),
\qquad
b_i \leftarrow b_i + u\, \mathrm{sign}(\bar{f} - f_i),
$$

where $\bar{f} = 1/E$ is the even share. The bias of an overloaded expert
falls, and that of an underloaded expert rises, which makes the
underloaded expert slightly easier to select. Since $b_i$ never enters the
weight $p_i$ in the equation, the model's loss contains no
balancing term, and no gradient competes with the language-modeling one.
The controller steers where tokens go; the gradients serve prediction
alone. The update rate sets how quickly the controller reacts. DeepSeek-V3
used $u = 0.001$ for the first 14.3 trillion of its training tokens; our
runs take 800 steps, and we use $u = 0.01$ so that the controller can
follow the router within a few dozen steps. DeepSeek-V3 also scores
experts with a sigmoid per expert instead of a softmax. Our layer keeps
the softmax and changes only the selection bias, the mechanism that
@Wang.Gao.Zeng.ea.2024 isolate.

### Balancing Methods Compared

Both remedies fit into an ordinary training loop. The loop below is
`d2l.train_steps` of that section, which draws minibatches, takes
optimizer steps, and returns the training losses, with the two remedies
added: `aux_weight` adds $\alpha$ times the sum of
$\mathcal{L}_{\mathrm{balance}}$ over the expert layers to the loss, and
`bias_rate` is the rate $u$ of the controller of the equation. The
controller is a method of the layer, `balance`, which runs after each
optimizer step on the load of that step's minibatch.

In [ ]:
@d2l.add_to_class(MoELayer)
def balance(self, rate):
    """The controller: move each bias one step toward the even load, then
    count the next batch's assignments afresh."""
    load = self.load[...]
    self.expert_bias[...] += rate * jnp.sign(load.mean() - load)
    self.load[...] = jnp.zeros_like(load)

@nnx.jit  # aux_weight and bias_rate are traced: one compilation for all
def moe_train_step(model, optimizer, X, Y, aux_weight, bias_rate):
    def loss_fn(model):
        ce = optax.softmax_cross_entropy_with_integer_labels(
            model(X), Y).mean()
        aux = sum(l.aux_loss[...] for l in moe_layers(model))
        return ce + aux_weight * aux, ce
    (_, ce), grads = nnx.value_and_grad(loss_fn, has_aux=True)(model)
    optimizer.update(model, grads)
    for l in moe_layers(model):  # The controller: not a gradient step
        l.balance(bias_rate)
    return ce

def train_moe_lm(model, data, optimizer, num_steps, aux_weight=0.0,
                 bias_rate=0.0):
    """d2l.train_steps with a balancing loss and the bias controller."""
    for l in moe_layers(model):
        l.reset_stats()
    losses = []
    while len(losses) < num_steps:
        for X, Y in data.train_dataloader():
            losses.append(moe_train_step(model, optimizer, jnp.asarray(X),
                                         jnp.asarray(Y), aux_weight,
                                         bias_rate))
            if len(losses) == num_steps:
                return [float(l) for l in jax.device_get(losses)]

To read a trained model, we compute its validation loss with
`d2l.lm_perplexity` of that section, whose logarithm is the mean
cross-entropy in nats per byte, and count during that pass how many
validation tokens each expert receives and how many assignments the
buffers drop. The shares of each layer are returned busiest first:

In [ ]:
def evaluate_moe(model, data):
    """Validation loss; each expert layer's shares of the validation
    tokens, busiest expert first; and the fraction of assignments dropped."""
    layers = moe_layers(model)
    for l in layers:
        l.reset_stats()
    loss = math.log(d2l.lm_perplexity(model, data.val_dataloader()))
    shares = [sorted((l.load / l.load.sum()).tolist(), reverse=True)
              for l in layers]
    dropped = sum(float(l.dropped) for l in layers) / sum(
        float(l.load.sum()) for l in layers)
    return loss, shares, dropped

def report(name, model, data):
    """Print the validation loss, the busiest expert's load in units of the
    even share, and the drops; return the shares."""
    val, shares, dropped = evaluate_moe(model, data)
    peak = max(l.num_experts * s[0]
               for l, s in zip(moe_layers(model), shares))
    print(f'{name:>15}: validation loss {val:.2f}, busiest expert at '
          f'{peak:.1f} times its share, {dropped:.0%} dropped')
    return shares

The experiments use the data and the training recipe of
that section. The data are the bytes of the Wikipedia articles of
`d2l.WikiText103`, about 270 million for training, in windows of 128 bytes
at random offsets, and the official validation split of about 1.1 million
bytes. A run almost never reads the same passage twice, so the validation
loss measures what a model has learned about English rather than how much
of the text it has memorized. The recipe, bundled as `d2l.LMOptimizer`, is
AdamW with weight decay on the weight matrices, a learning rate that rises
linearly over the first 100 steps and then follows a cosine to zero, and
clipping of the gradient norm at 1. Our model is a small GPT, two blocks
of width 128 with eight experts per block and one active per token, with the capacity factor
of 1.25. We first read a fresh model, then train three models for 800
steps each. The runs differ only in balancing: none, the auxiliary loss at
$\alpha = 0.01$, or the bias controller at $u = 0.01$. Each model draws
its own initialization.

In [ ]:
data = d2l.WikiText103(batch_size=64, num_steps=128)

def small_gpt(vocab_size, capacity_factor=None):
    """Two blocks of width 128; with a capacity factor, each feed-forward
    network becomes eight experts with one active per token."""
    experts = lambda rngs: MoELayer(128, 8, 1,
                                    capacity_factor=capacity_factor, rngs=rngs)
    return d2l.GPT(vocab_size, num_hiddens=128, num_heads=4, num_blks=2,
                   ffn_factory=None if capacity_factor is None else experts,
                   rngs=nnx.Rngs(d2l.get_key()))

`small_gpt` builds this model, and without a capacity factor the dense
GPT of the same size, which that section uses. Each line of the
output reads one model: the fresh one, then the three trained ones.

In [ ]:
shares = {'initialization': report(
    'initialization', small_gpt(len(data.vocab), capacity_factor=1.25), data)}
for name, aux_weight, bias_rate in (('no balancing', 0.0, 0.0),
                                    ('auxiliary loss', 0.01, 0.0),
                                    ('bias controller', 0.0, 0.01)):
    model = small_gpt(len(data.vocab), capacity_factor=1.25)
    optimizer = d2l.LMOptimizer(model, lr=2e-3, num_steps=800, warmup=100)
    train_moe_lm(model, data, optimizer, 800, aux_weight, bias_rate)
    shares[name] = report(name, model, data)

The plot shows the shares of the eight experts, sorted from the busiest to
the idlest and averaged over the two layers, against the even share of
$1/8$:

In [ ]:
ranks = list(range(1, 9))
mean_shares = [[sum(rank) / len(rank) for rank in zip(*s)]
               for s in shares.values()]  # Mean over the layers
d2l.plot(ranks, mean_shares + [[1 / 8] * 8], 'expert, busiest first',
         'share of tokens', legend=list(shares) + ['even share'],
         fmts=('C7x:', 'C0o-', 'C1s--', 'C2^-.', 'k-'), legend_loc='outside')

The fresh model is unbalanced before it has seen any data, unlike the
layer that we fed random vectors above, and its buffers drop many
assignments. At initialization the residual stream of the first block
carries little more than the embedding of the current byte. Most
occurrences of a byte therefore look alike to the router and go to the
same expert, and bytes are far from equally frequent: nearly one byte in
five is a space. In the second block, attention has added to every
position nearly the same average of the earlier bytes, and this
common offset tilts most tokens toward the same expert. Training without
balancing does not undo the imbalance. The routing stays concentrated: the
busiest expert of a layer receives about twice its even share or more, some
experts receive almost none, and the buffers keep dropping the tokens that
the busy experts have no room for.

Both remedies spread the tokens much more evenly and drop few assignments.
Both runs end at or below the validation loss of the unbalanced one, and
the further that run has collapsed, the wider the gap. In the unbalanced
model, experts that receive almost no tokens add parameters
that the model hardly uses, and many tokens skip their expert altogether.
At this scale neither remedy ends consistently ahead of the other. Their
difference shows in larger models: @Wang.Gao.Zeng.ea.2024 trained models of 1 and 3
billion parameters both ways, and the controller balanced the load much
more evenly than the auxiliary loss while reaching a slightly lower
validation perplexity.

## Mixture-of-Experts Language Models

Balancing makes the experts usable. Whether they make a better language
model, and how released models arrange them, is the subject of this last
part.

### A Mixture-of-Experts GPT

The expert layer was built to answer one question: at the same
computation per token, does a model with eight times the feed-forward
parameters predict text better? Extra parameters help only a model that
has more text than it can absorb, so we compare in that regime: the small
GPT of the balancing experiments, two blocks of width 128, trained on
about 50 million bytes, more than a hundred per parameter. The dense
`d2l.GPT` competes with the same model with each feed-forward network
replaced by eight experts, one active per token. The expert layers are
balanced by the auxiliary loss at $\alpha = 0.01$, as in the Switch
transformer, whose experiments we compare with below. Each expert is a
full-width `d2l.FeedForward`, so a token passes through as many parameters
in the mixture as in the dense model, apart from the router's
$128 \times 8$ weights per block.

The buffers are larger than in the balancing experiments: the capacity
factor is 2 rather than 1.25, another setting with which the Switch
transformer was trained [@fedus2022switch]. Bytes need the room. At
$c = 1.25$ an expert's buffer holds $1.25/8 \approx 16\%$ of the
assignments, yet nearly one byte in five is a space, and the first block,
which sees little more than the current byte, tends to send the
occurrences of a byte to the same expert. When one expert collects most of the spaces
together with another frequent byte, its buffer overflows. In our runs at
$c = 1.25$, the first block of some models dropped a tenth of its tokens
for a thousand steps or more. The longer the overflow lasted, the smaller
the mixture's final lead over the dense model, which ranged from almost
nothing to five hundredths of a nat. At $c = 2$ a buffer holds a quarter
of the assignments, and after the first few hundred steps almost no
assignment is dropped.

In [ ]:
models = {'dense': small_gpt(len(data.vocab)),
          'MoE': small_gpt(len(data.vocab), capacity_factor=2)}
for name, model in models.items():
    print(f'{name:>5}: {d2l.num_params(model) / 1e6:5.2f}M parameters, '
          f'{active_params(model) / 1e6:.2f}M active per token')

The mixture stores 5.3 times the parameters of the dense model and uses
the same number per token. Both models train with the recipe of
that section for 6,000 steps, and we measure the validation loss
every 250 steps:

In [ ]:
num_steps, every = 6000, 250
curves = {}
for name, model in models.items():
    optimizer = d2l.LMOptimizer(model, lr=2e-3, num_steps=num_steps,
                                warmup=100)
    curves[name] = []
    for _ in range(num_steps // every):
        train_moe_lm(model, data, optimizer, every, aux_weight=0.01)
        curves[name].append(
            math.log(d2l.lm_perplexity(model, data.val_dataloader())))
    print(f'{name:>5}: validation loss {curves[name][-1]:.2f} nats')
d2l.plot(list(range(every, num_steps + 1, every)), list(curves.values()),
         'step', 'validation loss (nats)', legend=list(curves),
         fmts=('-', '--'))

Early in training the mixture trails the dense model: each fresh
expert's output is scaled by a probability near $1/8$, so the expert
layers start out much weaker than the dense feed-forward networks, and
each expert learns from only an eighth of the tokens. As the dense model
runs out of capacity, its curve flattens, while the mixture keeps
improving. Its validation loss falls below the dense model's within the
first two thousand steps and stays below it to the end, by a few hundredths
of a nat.
At the same computation per token, the stored experts make a better
model. Large studies report the same effect. The Switch transformer reached the quality of a dense T5
model several times faster at the same number of operations per token, and
it already gained from two experts [@fedus2022switch].
@Clark.Casas.Guy.ea.2022 found that routing improved models of
every size they trained, from 15 million to 1.3 billion parameters, with
the largest gains for the smallest models.

The gain needs data. Each expert sees about $1/E$ of the tokens, and the
extra parameters help only if the text holds more than the dense model can
learn from it. A corpus read many times over offers less, and ours is
read less than once. Exercise 7 repeats the
comparison with the six-block GPT of that section, which at 1,000
steps has more parameters than its text can fill.
@Krajewski.Ludziejewski.Adamczewski.ea.2024 found that when the
training tokens grow with the model, the advantage of mixtures over dense
models grows with the training budget.

The gain also has costs that the operation count leaves out. The mixture
keeps 5.3 times the parameters in memory, together with AdamW's two state
values per parameter. Its step takes longer than its operation count
suggests: the buffers hold twice as many rows as there are tokens,
copying tokens into the buffers and outputs back moves data without
computing, and the experts' weights are stacked anew in every step. And
wherever the load is uneven, the buffers drop assignments, which a dense
model never does.

### Fine-Grained and Shared Experts

Released mixtures differ in how they divide a budget of expert parameters.
Mixtral's experts are full copies of the feed-forward network, eight per
layer and two per token. DeepSeekMoE [@Dai.Deng.Zhao.ea.2024] spends
such a budget differently, in the two steps shown in
the figure. Their example starts from 16 full-width
experts with 2 active per token. The first step cuts each expert into $m$
narrower ones and lets each token select $m$ times as many: with $m = 4$, 8
of 64 experts of a quarter width instead of 2 of 16 full-width ones.
Neither the stored nor the active parameter count changes. The number of
combinations a token can select does, from $\binom{16}{2} = 120$ to
$\binom{64}{8} \approx 4.4 \times 10^{9}$. A token can then assemble its
network from many narrow specialists, say one for chemical vocabulary and
another for nested clauses, instead of selecting two broad experts that
must each cover a wide range of inputs.

![Three ways to spend the same expert parameters. Each box is an expert, its width proportional to its size, and the highlighted experts are the ones selected for one token. Top: 16 experts, of which a token uses 2. Middle: the same parameters as 64 experts of a quarter width, of which a token uses 8. Bottom: one of the narrow experts becomes a shared expert that every token uses, and each token selects 7 of the other 63. All three rows store and use the same number of parameters per token.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-transformers-moe-fine.svg)

The second step makes one of the narrow experts *shared*: every token
passes through it, in addition to its routed experts. Some processing is
needed by every token, whatever its topic. With routed experts only, each
expert must learn that processing for itself, so the same function is
stored many times over. A shared expert stores it once and leaves the
routed experts free to differ. In experiments at 2 billion stored
parameters, a model with both refinements reached a lower loss on the Pile
than a GShard model with the same stored and active parameters (1.81
against 1.87), and about matched a GShard model with 1.5 times the expert
parameters and 1.5 times the computation [@Dai.Deng.Zhao.ea.2024].
Neither refinement changes the logic of our layer: narrower experts need
only a configurable width, and a shared expert is one more feed-forward
network whose output is added to the routed mixture. The exercises build
both.

Released models make different choices along these axes, as
the table shows.

<!-- tbl-caption:Routed-expert configurations of released MoE language models. -->

| Model | Experts per layer | Active $k$ | Expert width / $d$ | Shared expert |
|---|---|---|---|---|
| Mixtral 8x7B [@Jiang.Sablayrolles.Roux.ea.2024] | 8 | 2 | 3.5 | none |
| DeepSeek-V3 [@Liu.Feng.Xue.ea.2024] | 256 | 8 | 0.29 | 1 |
| Qwen3-235B [@Yang.Li.Yang.ea.2025] | 128 | 8 | 0.38 | none |
| gpt-oss-120b [@OpenAI.2025] | 128 | 4 | 1.0 | none |

The three later models use many more experts than Mixtral and between 4
and 8 active per token, and most of them make each expert narrower than
the model itself. They disagree on the shared expert: DeepSeek's models
have used shared experts since DeepSeekMoE, two in DeepSeek-V2
[@DeepSeek-AI.2024] and one in DeepSeek-V3, whereas Qwen3 uses none.
How many experts a model should have, how wide, and how many tokens it
needs before its stored parameters help are questions of scaling, of the
kind that that section asks for dense models.

## Summary

A dense transformer spends computation on every parameter for every token.
A mixture of experts replaces each block's feed-forward network by $E$
experts and a linear router that sends each token to $k$ of them, weighted
by the router's probabilities: the stored parameters grow with $E$, the
computation per token with $k$. An efficient layer copies each token into
a fixed-size buffer of each of its experts and runs all experts at once on
their buffers, at the cost of dropping the assignments that do not fit.
Left alone, routing concentrates on a few experts, both because similar
tokens go to the same expert and because routing reinforces itself. An auxiliary loss
on the product of load and probability, or a controller that biases only
the selection, evens out the load. With balanced experts, a GPT with eight
experts per block and one active per token reaches a lower validation loss
than the dense model at the same computation per token, once the dense
model has more text than it can absorb, while storing 5.3 times the
parameters; released models refine the design with many narrow
experts and a shared expert.

## Exercises

1. **Router gradient.** The top-$k$ selection in the equation has
   no gradient, yet the router trains. Consider one token with router logits
   $\mathbf{z} = \mathbf{W}_r \mathbf{x}$, so that
   $\mathbf{p} = \mathrm{softmax}(\mathbf{z})$, and treat the selected set
   $\mathcal{E}_k$ as a constant. Write the layer output as
   $\mathbf{y} = \sum_{i \in \mathcal{E}_k} p_i\, \mathrm{FFN}_i(\mathbf{x})$
   and let
   $s_i = (\partial \mathcal{L} / \partial \mathbf{y})^\top \mathrm{FFN}_i(\mathbf{x})$
   be the sensitivity of the loss to expert $i$'s output.
    1. Derive $\partial \mathcal{L} / \partial z_j$ for every expert $j$ in
       terms of $\mathbf{p}$ and the $s_i$. Which logits receive a nonzero
       gradient, and on which experts' outputs does the gradient of an
       unselected logit depend?
    1. Mixtral weights the selected experts by renormalized probabilities
       [@Jiang.Sablayrolles.Roux.ea.2024],

       $$\tilde{p}_i = \frac{p_i}{\sum_{j \in \mathcal{E}_k} p_j}, \qquad i \in \mathcal{E}_k.$$

       Show that for $k = 1$ the router receives no gradient from the
       language-modeling loss.
    1. Show that for $k = 2$ with renormalization the output depends on
       the logits only through the difference $z_a - z_b$ of the two
       selected experts $a$ and $b$. Which routing decision can the router
       learn from the loss in this case, and which can it not?

1. **Serving arithmetic.** Qwen3-235B-A22B [@Yang.Li.Yang.ea.2025]
   has $d = 4096$ and 94 layers. Every layer holds 128 SwiGLU experts of
   width 1536, of which each token activates 8, and no shared expert. Its
   attention has 64 query heads and 4 key--value heads, each of dimension
   128, and its untied embedding and output matrices cover a vocabulary of
   151,936 tokens.
    1. Use `moe_accounting` for the experts and the router, and add the
       attention and embedding parameters as
       that section does for Mixtral. Reproduce the
       235B stored and 22B active parameters of the model's name, ignoring
       normalization weights.
    1. Decoding a batch of $B$ sequences sends one token per sequence
       through every layer, and its speed is set by the bytes of weights
       read per step (that section). Assume that each token
       selects a uniformly random set of 8 experts, independently of the
       other tokens. Derive the expected number of distinct experts that a
       layer reads, and evaluate it for $B = 1$, 8, and 64. Above which
       batch size does a step read 90% of the expert weights, and which
       dense model then reads the same number of bytes per step?

1. **Balancing-loss gradient.** The gradient of the equation flows
   through $\bar{p}_i$ only, because $f_i$ is a count. Consider a batch of
   $N$ tokens, and write $\mathbf{z}(\mathbf{x})$ for the router logits of
   token $\mathbf{x}$.
    1. Derive
       $\partial \mathcal{L}_{\mathrm{balance}} / \partial z_j(\mathbf{x})$.
       For which experts $j$ does the gradient step lower the logit of
       token $\mathbf{x}$, and for which does it raise it?
    1. Show that the gradient vanishes for every token whenever the batch
       load is uniform, $f_i = 1/E$, whatever the individual distributions
       $\mathbf{p}(\mathbf{x})$ are. Which property of the router does the
       loss therefore leave unconstrained?
    1. Show that the gradient for a token tends to zero as its router
       distribution approaches a one-hot vector. What does this imply for
       the ability of the loss to repair a collapse in which the router
       has become confident?

1. [code] **Feedback-loop ablations.** The argument of
   that section names two reinforcing effects: a
   frequently selected expert improves on its tokens, and the router
   gradient raises its selection probability.
    1. Stop the gradient at the router logits inside `MoELayer`, so that
       $\mathbf{W}_r$ keeps its initial values while every other parameter
       trains. Predict whether the unbalanced run of
       that section still concentrates its
       routing, then rerun it and compare the busiest expert's share. What
       still changes the routing when $\mathbf{W}_r$ is fixed?
    1. Restore the router gradient and rerun the three runs with
       `num_active=2`. Does the unbalanced run still concentrate, and as
       strongly? Use the router gradient of
       problem 1 to explain how a second
       active expert changes the signal that the runner-up receives.

1. [code] **Bias controller in isolation.** The controller of
   the equation can be studied without training. Build
   `MoELayer(256, num_experts=8, num_active=1)`, draw a fixed batch of
   shape $(4, 256, 256)$, that is, 1024 standard-normal tokens in four
   sequences, and add one fixed standard-normal vector
   $\boldsymbol{\mu} \in \mathbb{R}^{256}$ to every token, which skews the
   untrained router toward a few experts. Then call `reset_stats` and
   repeat 500 times: run the layer on the batch, and call its `balance`
   method, which updates the biases and resets `load`.
    1. For $u \in \{0.001, 0.01, 0.1\}$, plot the largest load
       $\max_i f_i$ against the iteration. How does the number of
       iterations needed to bring it below $1.5/E$ scale with $u$, and how
       does the load behave afterwards?
    1. After the last iteration, measure the fraction of tokens whose
       selected expert differs from the expert with the largest
       $p_i(\mathbf{x})$. What determines this fraction, and how does it
       depend on $u$?
    1. Replace the sign in the equation by the raw difference,
       $b_i \leftarrow b_i + u' (\bar{f} - f_i)$, with $u'$ chosen so that
       the largest first step equals that of $u = 0.01$. Which behavior
       from the first sub-problem changes?

1. **Capacity factor.** ● Our layer gives each expert a buffer of
   $C = c\, Nk/E$ slots for a batch of $N$ tokens, as GShard and the Switch
   transformer do [@Lepikhin.Lee.Xu.ea.2021; @fedus2022switch], and
   drops the assignments that do not fit.
    1. Without compilation, a layer could size its buffers by each
       batch's routing. Explain why the buffer size must still be fixed in
       advance when the experts live on different devices.
    1. Take $k = 1$, $N = 4096$, and a router that sends each token to a
       uniformly random expert, independently of the other tokens. The
       load of one expert is then binomial with mean $N/E$. Use a normal
       approximation, together with $\mathbb{E}[\max(Z, 0)] = 1/\sqrt{2\pi}$
       for a standard normal $Z$, to estimate the fraction of tokens
       dropped at $c = 1$ for $E = 8$ and for $E = 64$. How does the
       fraction scale with $E$ and $N$, and what does $c = 1.25$ change?
    1. Our layer fills the buffers in the order of the tokens. In a
       causal language model, can whether a token is dropped depend on
       later tokens of its own sequence? On other sequences of the
       minibatch? What does this imply for a model that is evaluated
       with batches of a different size or composition than in training,
       for example while decoding one token per sequence?

1. [code] **When the mixture loses.** that section compares
   the two models where the dense model has more text than it can absorb.
    1. Repeat the comparison with the six-block GPT of width 256 from
       that section and its budget of 1,000 steps, with eight
       full-width experts per block. Train each model three times. Which
       one ends lower?
    1. Compute the training bytes per dense parameter in both comparisons,
       and explain the outcome with the scaling laws of
       that section and the data each expert sees.
    1. Train the six-block pair for longer. Does the mixture overtake the
       dense model, and if so, after how many steps?

1. [code] **Shared expert.** ● Write `SharedMoELayer`, a subclass of
   `MoELayer` that adds the output of one extra `d2l.FeedForward`, applied
   to every token with weight one, to the routed
   mixture (that section). Compare two models in the
   setup of that section, both balanced by the
   bias controller: `MoELayer` with $E = 8$ and $k = 2$, and
   `SharedMoELayer` with 7 routed experts and $k = 1$. Both store eight
   experts per layer and activate two per token.
    1. Train each model three times and compare their validation losses.
       Is the difference larger than the spread between the runs of one
       model?
    1. @Dai.Deng.Zhao.ea.2024 test whether a shared expert holds
       processing that the routed experts lack: they disable it and
       activate one more routed expert, which keeps the per-token
       computation fixed. Repeat this test on a trained `SharedMoELayer`
       model. Remove the shared path, set `num_active` to 2 in every layer,
       and compare the validation loss with that of the intact model. What
       does the result say about the division of labor? Why does removing
       the shared path without the replacement say little about it?

1. [extended] **Expert granularity.** ● that section
   cites large-scale evidence that narrower, more numerous experts help at
   a fixed budget, and this section does not test it. In the setup of
   that section, an expert at $d = 128$ has
   SwiGLU width 341. Give `MoELayer` experts of configurable width, and
   compare $(E, k) \in \{(8, 1), (16, 2), (32, 4), (64, 8)\}$ with expert
   widths 341, 171, 85, and 43, all balanced by the bias controller. These
   configurations store and activate the same expert parameters as
   $(8, 1)$ to within 1%, while the number of expert combinations
   available to a token grows from 8 to about $4.4 \times 10^9$.
    1. Train each configuration three times and compare the validation
       losses. Does finer granularity help at this scale by more than the
       spread between runs? Relate the outcome to the ablations of
       @Dai.Deng.Zhao.ea.2024 .
    1. The argument of that section assumes that
       tokens use many different combinations. In one layer of each trained
       model, record the selected expert set of every token in a fixed set
       of validation batches, and count the distinct sets that occur.
       Compare the count with the number of tokens and with the number of
       available combinations. How much of the added flexibility does the
       model use?

[Discussions](https://d2l.discourse.group/t/36779)